# Phase 1: Market Data and Returns

## Project context

This notebook starts the Portfolio Optimization and Factor Research Lab by building the market data foundation for later CAPM, factor research, portfolio optimization, and backtesting work. Phase 1 focuses only on price loading, return calculation, risk and return summary, and benchmark comparison.

## Ticker universe and benchmark

- Asset universe: AAPL, MSFT, JPM, PG, XOM, JNJ, KO, NVDA
- Benchmark: SPY
- Period: 2019-01-01 to latest available yfinance data
- Data source: Yahoo Finance via `yfinance`

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR, OUTPUTS_DIR, FIGURES_DIR
from src.data_loader import download_price_data, extract_adjusted_close, save_price_data
from src.returns import (
    calculate_daily_returns,
    calculate_monthly_returns,
    calculate_max_drawdown,
    summarize_asset_performance,
)
from src.visualization import (
    plot_price_history,
    plot_cumulative_returns,
    plot_correlation_heatmap,
    plot_risk_return_scatter,
)

TICKERS = ["AAPL", "MSFT", "JPM", "PG", "XOM", "JNJ", "KO", "NVDA"]
BENCHMARK = "SPY"
ALL_TICKERS = TICKERS + [BENCHMARK]
START_DATE = "2019-01-01"

RETURNS_DIR = OUTPUTS_DIR / "returns"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
RETURNS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def _normalize(values):
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    if not finite.any():
        return np.zeros_like(values)
    vmin = np.nanmin(values[finite])
    vmax = np.nanmax(values[finite])
    if np.isclose(vmin, vmax):
        return np.full_like(values, 0.5, dtype=float)
    return (values - vmin) / (vmax - vmin)


def save_plotly_or_pillow(fig, output_path, chart_type, data):
    """Save a Plotly figure as PNG, with a data-driven Pillow fallback if Kaleido is unavailable."""
    output_path = Path(output_path)
    try:
        fig.write_image(str(output_path), width=1200, height=700, scale=2)
        return "plotly"
    except Exception as exc:
        draw_basic_png(output_path, chart_type, data, str(exc))
        return "pillow_fallback"


def draw_basic_png(output_path, chart_type, data, reason):
    width, height = 1200, 700
    margin = 80
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    title_font = ImageFont.load_default()
    text_font = ImageFont.load_default()
    draw.text((margin, 24), output_path.stem.replace("_", " ").title(), fill="black", font=title_font)
    draw.text((margin, 48), "Rendered with Pillow fallback because Plotly PNG export was unavailable.", fill="#555555", font=text_font)

    left, top, right, bottom = margin, 110, width - margin, height - margin
    draw.rectangle((left, top, right, bottom), outline="#333333")

    if chart_type == "lines":
        frame = data.dropna(how="all")
        if len(frame) > 260:
            frame = frame.iloc[np.linspace(0, len(frame) - 1, 260).astype(int)]
        colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#17becf"]
        for idx, col in enumerate(frame.columns):
            series = frame[col].astype(float)
            y_norm = _normalize(series.values)
            points = []
            for i, value in enumerate(y_norm):
                x = left + (right - left) * i / max(len(y_norm) - 1, 1)
                y = bottom - (bottom - top) * value
                points.append((x, y))
            if len(points) > 1:
                draw.line(points, fill=colors[idx % len(colors)], width=2)
            draw.text((right - 130, top + 18 * idx), str(col), fill=colors[idx % len(colors)], font=text_font)
    elif chart_type == "heatmap":
        matrix = data.astype(float)
        labels = list(matrix.columns)
        n = len(labels)
        cell = min((right - left) / max(n, 1), (bottom - top) / max(n, 1))
        for i, row in enumerate(labels):
            for j, col in enumerate(labels):
                value = matrix.loc[row, col]
                red = int(255 * max(value, 0))
                blue = int(255 * abs(min(value, 0)))
                green = int(230 * (1 - abs(value)))
                x0 = left + j * cell
                y0 = top + i * cell
                draw.rectangle((x0, y0, x0 + cell, y0 + cell), fill=(red, green, blue), outline="white")
                draw.text((x0 + 4, y0 + 4), f"{value:.2f}", fill="black", font=text_font)
        for i, label in enumerate(labels):
            draw.text((left + i * cell + 4, top - 18), label, fill="black", font=text_font)
            draw.text((left - 50, top + i * cell + 4), label, fill="black", font=text_font)
    elif chart_type == "scatter":
        frame = data.copy()
        x = _normalize(frame["annualized_volatility"].values)
        y = _normalize(frame["annualized_return"].values)
        for i, ticker in enumerate(frame.index):
            px = left + (right - left) * x[i]
            py = bottom - (bottom - top) * y[i]
            draw.ellipse((px - 5, py - 5, px + 5, py + 5), fill="#1f77b4")
            draw.text((px + 8, py - 8), str(ticker), fill="black", font=text_font)

    image.save(output_path)


## Market data download

In [ ]:
raw_data = download_price_data(ALL_TICKERS, start_date=START_DATE)
prices = extract_adjusted_close(raw_data)
prices = prices.reindex(columns=ALL_TICKERS)
prices = prices.dropna(how="all")

missing_tickers = [ticker for ticker in ALL_TICKERS if ticker not in prices.columns or prices[ticker].dropna().empty]
if missing_tickers:
    raise ValueError(f"Missing downloaded price data for: {missing_tickers}")

price_output_path = save_price_data(prices, PROCESSED_DATA_DIR / "adjusted_close_prices.csv")
price_output_path

## Adjusted close price review

In [ ]:
display(prices.head())
display(prices.tail())
print(f"Price data shape: {prices.shape}")
print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")

## Daily returns

In [ ]:
daily_returns = calculate_daily_returns(prices)
daily_returns.to_csv(RETURNS_DIR / "daily_returns.csv", index_label="date")
display(daily_returns.head())
print(f"Daily returns shape: {daily_returns.shape}")

## Monthly returns

In [ ]:
monthly_returns = calculate_monthly_returns(prices)
monthly_returns.to_csv(RETURNS_DIR / "monthly_returns.csv", index_label="date")
display(monthly_returns.head())
print(f"Monthly returns shape: {monthly_returns.shape}")

## Annualized risk and return summary

In [ ]:
performance_summary = summarize_asset_performance(daily_returns)
performance_summary.to_csv(RETURNS_DIR / "asset_performance_summary.csv", index_label="ticker")
display(performance_summary.style.format({
    "annualized_return": "{:.2%}",
    "annualized_volatility": "{:.2%}",
    "sharpe_ratio": "{:.2f}",
    "max_drawdown": "{:.2%}",
}))

## Correlation analysis

In [ ]:
correlation_matrix = daily_returns.corr()
correlation_matrix.to_csv(RETURNS_DIR / "correlation_matrix.csv", index_label="ticker")
display(correlation_matrix.style.format("{:.2f}"))

## Drawdown analysis

In [ ]:
drawdown_summary = calculate_max_drawdown(daily_returns).sort_values()
display(drawdown_summary.to_frame("max_drawdown").style.format("{:.2%}"))

## Benchmark comparison against SPY

In [ ]:
benchmark_metrics = performance_summary.loc[[BENCHMARK]].rename(index={BENCHMARK: "Benchmark: SPY"})
asset_metrics = performance_summary.loc[TICKERS]
benchmark_comparison = asset_metrics.assign(
    excess_annualized_return=asset_metrics["annualized_return"] - performance_summary.loc[BENCHMARK, "annualized_return"],
    volatility_gap=asset_metrics["annualized_volatility"] - performance_summary.loc[BENCHMARK, "annualized_volatility"],
)
display(benchmark_metrics.style.format("{:.2%}", subset=["annualized_return", "annualized_volatility", "max_drawdown"]))
display(benchmark_comparison.sort_values("excess_annualized_return", ascending=False).style.format({
    "annualized_return": "{:.2%}",
    "annualized_volatility": "{:.2%}",
    "sharpe_ratio": "{:.2f}",
    "max_drawdown": "{:.2%}",
    "excess_annualized_return": "{:.2%}",
    "volatility_gap": "{:.2%}",
}))

## Figures

In [ ]:
figures = {
    "price_history.png": (plot_price_history(prices), "lines", prices),
    "cumulative_returns.png": (plot_cumulative_returns(daily_returns), "lines", (1 + daily_returns.fillna(0)).cumprod() - 1),
    "correlation_heatmap.png": (plot_correlation_heatmap(daily_returns), "heatmap", correlation_matrix),
    "risk_return_scatter.png": (plot_risk_return_scatter(performance_summary), "scatter", performance_summary),
}

figure_export_methods = {}
for filename, (fig, chart_type, data) in figures.items():
    method = save_plotly_or_pillow(fig, FIGURES_DIR / filename, chart_type, data)
    figure_export_methods[filename] = method

figure_export_methods

## Initial business interpretation

- This universe combines large-cap technology, financials, consumer staples, energy, healthcare, beverages, and semiconductor exposure.
- SPY provides a broad U.S. equity benchmark for comparing asset-level return and risk.
- The risk-return summary, drawdown table, and correlation matrix identify which assets have historically contributed higher return, higher volatility, and higher diversification potential since 2019.
- These outputs are descriptive research inputs only and are not investment recommendations.

## Phase 1 limitations

- Yahoo Finance data can change because of vendor revisions, ticker availability, and corporate action adjustments.
- Returns are historical and do not imply future performance.
- No CAPM, factor model, portfolio optimization, transaction cost model, or backtest has been built in this phase.
- The risk-free rate is held at zero for the initial Sharpe ratio summary and will be refined later if needed.

## Next steps for Phase 2 CAPM and factor research

- Estimate benchmark beta and alpha for each asset using SPY as the market proxy.
- Add factor research inputs after the return foundation is stable.
- Compare asset behavior across market cycles and identify financially explainable drivers.